# hAQT — Cloud / Colab demo run

Use this notebook on a Deep Learning VM or Colab GPU **after** the local pipeline is green.

Goals: final multi-precision metrics for Gemma-2 2B + 9B, screenshots, short demo recording.

In [ ]:
# %pip install -r requirements.txt
from pathlib import Path
from src.data_loader import load_benchmark_dataset
from src.quantizer import LoadConfig, ModelSize, Precision, recommended_precisions

records = load_benchmark_dataset("data/cve_sample.json")
print(f"loaded {len(records)} sample CVEs")
print("2B precisions @16GB:", recommended_precisions(ModelSize.GEMMA2_2B, 16))
print("9B precisions @16GB:", recommended_precisions(ModelSize.GEMMA2_9B, 16))
print("9B precisions @40GB:", recommended_precisions(ModelSize.GEMMA2_9B, 40))

In [ ]:
# End-to-end matrix → outputs/ (same path as scripts/run_benchmark.py)
# Prefer the CLI on a VM: python scripts/run_benchmark.py --limit 8 --models 2b,9b --vram-gb 40

from src.profiler import profile_prompt_batch
from src.results import save_run
from src.scoring import reports_to_flat_metrics, score_predictions
from src.tasks import build_all_examples

examples = build_all_examples(records)[:8]
prompts = [e.prompt for e in examples]
print(f"{len(examples)} prompts")

configs = [
    LoadConfig(ModelSize.GEMMA2_2B, p)
    for p in recommended_precisions(ModelSize.GEMMA2_2B, 16)
]
# Uncomment on large cloud GPUs:
# configs += [
#     LoadConfig(ModelSize.GEMMA2_9B, p)
#     for p in recommended_precisions(ModelSize.GEMMA2_9B, 40)
# ]

rows = []
for cfg in configs:
    print(f"Running {cfg.model_size.value}/{cfg.precision.value} …")
    result = profile_prompt_batch(cfg, prompts, max_new_tokens=64)
    m = result.metrics
    if result.generations and len(result.generations) == len(examples):
        m.task_scores = reports_to_flat_metrics(
            score_predictions(examples, result.generations)
        )
    rows.append(m)
    print(m.to_dict())

path = save_run(rows, "outputs", meta={"source": "demo_run.ipynb", "n_examples": len(examples)})
print("wrote", path)